# Language, Writing, and AI — Coastal Carolina Workshop

This notebook accompanies a 90-minute session for English students. You will try two activities and watch an instructor-led prediction demo. No coding experience is required.

| Tier | Task | Format |
|---|---|---|
| 1 | Extract entities and likely actions | Hands-on |
| 2 | Evaluate text with rules and revise it | Hands-on |
| 3 | Learn patterns from synthetic outcomes | Instructor demo |

Run cells in order with **Shift + Enter** or the play button. Keep a copy in Google Drive to save edits. Use only text you can share; all supplied examples are synthetic.

Copyright (c) 2026 David Young. Code cells: [MIT](https://github.com/dyoung8989/CoastCarolinaWorkshop/blob/main/LICENSE-MIT). Markdown explanations and standalone sample teaching text: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). See [licensing details](https://github.com/dyoung8989/CoastCarolinaWorkshop/blob/main/LICENSE.md) for scope and attribution. Third-party dependencies retain their own licenses.

## Step 0: Setup

Run these two cells first. Setup downloads the libraries and a small English model; allow a few minutes. If Colab requests a restart, restart the session and run from the top. If the download fails, pair with someone or follow the instructor's saved demo.

In [ ]:
# Install the versions used by this workshop into the Colab session.
%pip install spacy==3.8.16 weasel==1.0.0 typer==0.27.3 click==8.4.2 scikit-learn==1.7.2 pandas==2.2.3 numpy==2.2.6 matplotlib==3.10.7
%pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

In [ ]:
import re
import spacy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

nlp = spacy.load("en_core_web_sm")

print("Setup complete! You're ready for Tier 1.")

---
## Tier 1: Extraction — what can the machine already find?

The simplest thing NLP can do is pull structured information out of unstructured text: names of things, key terms, action items. No judgment involved — just finding and labeling.

Here's a short sample procedure, the kind of thing a technical writer might produce:

In [ ]:
sample_doc_1 = """
Before you begin, make sure you have administrator access to the instance. First, navigate
to System Settings and select User Administration. You must verify that the user's role
includes "admin" before proceeding. Next, click Add User and enter the required fields:
username, email, and department. The system should automatically assign a default password,
but you can override this if needed. Please confirm the new account by clicking Save. If the
account does not appear in the list within 60 seconds, contact the IT Help Desk at extension
4521. Do not share admin credentials with other users under any circumstances.
"""

print(sample_doc_1)

### Try it: find the named things

spaCy (the library we loaded) can scan text and label the proper nouns and named things it recognizes — organizations, products, quantities, and more.

In [ ]:
doc = nlp(sample_doc_1)

print("Named things spaCy found:\n")
for ent in doc.ents:
    print(f"  {ent.text!r:30}  ->  {ent.label_}")

### Try it: find the action items

This one's a simple heuristic, not magic: we look for sentences that contain words a procedure typically uses to tell someone to *do* something ("must", "click", "please", "do not"...). It's a taste of how a lot of real-world text tooling starts — simple pattern matching, no machine learning required yet.

In [ ]:
action_keywords = ["must", "click", "please", "do not", "ensure", "verify", "confirm", "contact", "navigate", "select"]

def extract_actions(text):
    # Match whole words/phrases, so "mustard" does not match "must".
    pattern = r"\b(?:" + "|".join(re.escape(word) for word in action_keywords) + r")\b"
    return [sent.text.strip() for sent in nlp(text).sents
            if re.search(pattern, sent.text, flags=re.IGNORECASE)]

print("Likely action items (keyword guesses):\n")
for action in extract_actions(sample_doc_1):
    print(f"  • {action}")

**Your turn:** replace the paragraph below, keeping the triple quotes, then run the cell. It analyzes your new text directly. Challenge one entity label and look for an action the keyword list missed. The entity recognizer uses a trained model; the action detector uses rules.

In [ ]:
your_text = """
Please send the draft to the campus writing center. Maya will review it on Friday.
"""

print("Entities:")
for entity in nlp(your_text).ents:
    print(f"  {entity.text!r} -> {entity.label_}")
print("\nLikely actions:")
for action in extract_actions(your_text):
    print(f"  • {action}")

---
## Tier 2: Automated evaluation — is this document ready for a reader?

Extraction just finds things. This tier goes a step further: it **judges** the text, combining a few measurable signals into one "document readiness" score — and tells you specifically what to fix.

It's still rule-based — we decide the thresholds below — but it's a real tool people build and use, not just a demo.

The signals we'll combine:
- **Readability** — how hard the sentences are to read (via the Flesch Reading Ease score)
- **Passive voice** — what share of sentences use passive voice ("the form is submitted" vs. "you submit the form")
- **Sentence length** — average words per sentence
- **Structure** — does the document have headers or numbered steps at all?

In [ ]:
def count_syllables(word):
    """Rough syllable count: counts groups of vowels. Good enough for a readability estimate."""
    word = word.lower()
    groups = re.findall(r"[aeiouy]+", word)
    count = len(groups)
    if word.endswith("e") and count > 1:
        count -= 1
    return max(count, 1)


def flesch_reading_ease(text):
    """A simplified version of the classic Flesch Reading Ease formula. Higher = easier to read."""
    words = re.findall(r"[a-zA-Z']+", text)
    sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    num_words = max(len(words), 1)
    num_sentences = max(len(sentences), 1)
    num_syllables = sum(count_syllables(w) for w in words)
    return 206.835 - 1.015 * (num_words / num_sentences) - 84.6 * (num_syllables / num_words)


def score_doc(text):
    """Scores a document's readiness for a reader, 0-100, with specific flags."""
    if not isinstance(text, str) or not re.search(r"[a-zA-Z]", text):
        raise ValueError("Enter a paragraph with English words before scoring.")
    doc = nlp(text)
    sentences = list(doc.sents)
    num_sentences = max(len(sentences), 1)

    # Passive voice: spaCy tags the subject of a passive sentence as 'nsubjpass'
    passive_count = sum(1 for sent in sentences if any(tok.dep_ == "nsubjpass" for tok in sent))
    passive_pct = passive_count / num_sentences * 100

    # Average sentence length, in words
    avg_sentence_length = sum(len([t for t in sent if t.is_alpha]) for sent in sentences) / num_sentences

    # Readability: higher = easier (roughly 0-100 scale)
    readability = flesch_reading_ease(text)

    # Structure: any markdown-style header or numbered step?
    has_structure = bool(re.search(r"(^#{1,3}\s)|(^\s*\d+[.)]\s)", text, re.MULTILINE))

    flags = []
    if readability < 50:
        flags.append("Readability is low — try shorter sentences and simpler words.")
    if passive_pct > 30:
        flags.append(f"{passive_pct:.0f}% of sentences are passive — consider rewriting in active voice.")
    if avg_sentence_length > 25:
        flags.append("Average sentence length is high — break up long sentences.")
    if not has_structure:
        flags.append("No headers or numbered steps detected — consider adding structure.")

    score = 100
    score -= max(0, 60 - readability)
    score -= passive_pct * 0.5
    score -= max(0, avg_sentence_length - 20) * 2
    score -= 0 if has_structure else 15
    score = max(0, min(100, round(score)))

    return {
        "readiness_score": score,
        "readability": round(readability, 1),
        "passive_voice_pct": round(passive_pct, 1),
        "avg_sentence_length": round(avg_sentence_length, 1),
        "has_structure": has_structure,
        "flags": flags,
    }

print("score_doc() is ready.")

### Try it: score a messy document

In [ ]:
messy_doc = """
The configuration settings are modified by the administrator once the new environment has
been provisioned by the infrastructure team. It should be noted that the deployment process,
which involves several interdependent steps that must be completed in a specific order to
avoid conflicts with existing services, is not always fully understood by new team members,
and as a result, errors are frequently introduced during the initial setup phase, which can
then cascade into larger system failures if they are not caught early by the monitoring tools
that have been configured to alert the on-call engineer.
"""

result = score_doc(messy_doc)

print(f"Readiness score: {result['readiness_score']} / 100\n")
print(f"  Readability:          {result['readability']}")
print(f"  Passive voice:        {result['passive_voice_pct']}%")
print(f"  Avg sentence length:  {result['avg_sentence_length']} words")
print(f"  Has structure:        {result['has_structure']}\n")
print("Flags:")
for flag in result["flags"]:
    print(f"  \u26a0 {flag}")

### Compare a rewrite

This rewrite is shorter and more structured, but it also omits detail. Compare both the score and what a reader would learn. Does a higher score guarantee a better procedure? These rules cannot judge completeness or correctness.

In [ ]:
improved_doc = """
## Deployment Steps

1. The infrastructure team provisions the new environment.
2. The administrator updates the configuration settings.
3. Complete each step in order to avoid conflicts with existing services.
4. Monitoring tools alert the on-call engineer if an error occurs.
"""

result2 = score_doc(improved_doc)

print(f"Readiness score: {result2['readiness_score']} / 100  (was {result['readiness_score']} / 100)\n")
print(f"  Readability:          {result2['readability']}")
print(f"  Passive voice:        {result2['passive_voice_pct']}%")
print(f"  Avg sentence length:  {result2['avg_sentence_length']} words")
print(f"  Has structure:        {result2['has_structure']}")

**Your turn:** change one sentence in `revision` below and rerun. Preserve the original meaning. Did the score reward an edit that also helps the intended reader? Passive voice is sometimes appropriate, and technical vocabulary may be necessary.

In [ ]:
# Edit the paragraph or replace it with your own non-sensitive text.
revision = messy_doc
revision_result = score_doc(revision)
print("Before:", result["readiness_score"], "After:", revision_result["readiness_score"])
print("Flags:", revision_result["flags"])


---
## Tier 3: Prediction — instructor-led demo

Now the instructor will train a model on synthetic past documents and their invented support-ticket outcomes. We will hold some examples out of training to see how the model handles unseen examples.

The data generator deliberately builds a relationship into the examples. Recovering that relationship demonstrates a method; it does not discover a fact about real documentation or establish that writing features cause support tickets.

In [ ]:
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(42)
n = 80

# Simulated features for 80 historical documents
readability = rng.normal(55, 20, n).clip(0, 100)
passive_voice_pct = rng.normal(20, 15, n).clip(0, 100)
avg_sentence_length = rng.normal(18, 8, n).clip(5, 50)
doc_age_days = rng.integers(10, 1000, n)
has_structure = rng.integers(0, 2, n)

# The "true" underlying risk pattern -- harder to read, more passive, older, and
# unstructured docs are more likely to generate support tickets. The model below
# never sees this formula; it has to learn the pattern from the data alone.
risk_signal = (
    (60 - readability) * 0.05
    + passive_voice_pct * 0.03
    + doc_age_days * 0.001
    - has_structure * 1.5
    + rng.normal(0, 1, n)
)
had_support_tickets = (risk_signal > np.median(risk_signal)).astype(int)

history = pd.DataFrame({
    "readability": readability,
    "passive_voice_pct": passive_voice_pct,
    "avg_sentence_length": avg_sentence_length,
    "doc_age_days": doc_age_days,
    "has_structure": has_structure,
    "had_support_tickets": had_support_tickets,
})

history.head(10)

### Train a model on what happened in the past

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

features = ["readability", "passive_voice_pct", "avg_sentence_length", "doc_age_days", "has_structure"]
X = history[features]
y = history["had_support_tickets"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Learn scaling from the training data only, then fit the classifier.
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
model.fit(X_train, y_train)
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
test_probabilities = model.predict_proba(X_test)[:, 1]
print(f"Held out {len(y_test)} synthetic examples before training.")
print(f"Test accuracy: {accuracy_score(y_test, model.predict(X_test)):.0%}")
print(f"Majority baseline accuracy: {accuracy_score(y_test, baseline.predict(X_test)):.0%}")
print(f"Test ROC AUC: {roc_auc_score(y_test, test_probabilities):.2f}")
print("\nAssociations in standardized feature units (not causal effects):")
weights = model.named_steps["logisticregression"].coef_[0]
for name, coef in sorted(zip(features, weights), key=lambda pair: abs(pair[1]), reverse=True):
    direction = "increases" if coef > 0 else "decreases"
    print(f"  {name:20} {direction} predicted risk (weight: {coef:+.3f})")

Accuracy is the fraction of correct binary predictions at the default threshold. The baseline always picks the most common training label. ROC AUC describes ranking across thresholds (0.5 is chance-level ranking; 1.0 is perfect ranking).

These 20 test examples are small and synthetic. The probabilities have not been shown to match real-world event rates. Coefficients summarize learned associations on comparable scales, but correlated features can still complicate interpretation.

### Rank three invented current-document examples

The feature values below are made up for illustration; they are not extracted from real documents.

In [ ]:
current_docs = pd.DataFrame({
    "name": ["New Quick-Start Guide", "Legacy API Reference (2019)", "Mid-tier Troubleshooting Doc"],
    "readability": [68, 32, 50],
    "passive_voice_pct": [10, 45, 25],
    "avg_sentence_length": [14, 28, 20],
    "doc_age_days": [20, 900, 400],
    "has_structure": [1, 0, 1],
})

risk_probabilities = model.predict_proba(current_docs[features])[:, 1]
current_docs["predicted_risk"] = (risk_probabilities * 100).round(1)

current_docs[["name", "predicted_risk"]].sort_values("predicted_risk", ascending=False)

### See it as a chart

In [ ]:
ranked = current_docs.sort_values("predicted_risk")

fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.barh(ranked["name"], ranked["predicted_risk"], color="#2a78d6", height=0.5)

for bar, value in zip(bars, ranked["predicted_risk"]):
    ax.text(min(value + 2, 98), bar.get_y() + bar.get_height() / 2, f"{value:.0f}%",
            va="center", ha="right" if value > 92 else "left", color="#0b0b0b", fontsize=10)

ax.set_xlim(0, 100)
ax.set_xlabel("Model probability of synthetic ticket outcome (%)", color="#52514e")
ax.set_title("Synthetic documentation risk demo", color="#0b0b0b", loc="left", fontsize=12)
ax.spines[["top", "right", "bottom"]].set_visible(False)
ax.spines["left"].set_color("#c3c2b7")
ax.tick_params(axis="both", colors="#52514e", length=0)
ax.set_xticks([])

plt.tight_layout()
plt.show()

**Discuss:** does the ranking match your intuition? What would you want to know about a document that this model *doesn't* capture (tone, audience, how critical the task is)?

---
## Wrap-up

You used extraction and rule-based evaluation, then watched a model learn a pattern from simulated outcomes. The tasks illustrate ways writing expertise can contribute to content operations, documentation quality, and collaboration with data teams.

Discuss: what did the tools miss? What knowledge did you bring as a reader and writer? What would you need to measure before using a ranking to prioritize work?

Next step: apply one exercise to a public paragraph and record both its useful findings and its mistakes. See the repository's student guide and resources for follow-up links.